<a href="https://colab.research.google.com/github/niranthkn2116-ui/BikeVision-AI/blob/main/bike_vission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# 🏍️ BIKEVISION AI - COMPLETE GOOGLE COLAB PROJECT
# Bike Image Classification | Gradio | Hugging Face CLIP
# ============================================================

!pip install -q gradio transformers torch pillow pandas

import gradio as gr
import torch
import pandas as pd
import tempfile
import html
from datetime import datetime
from transformers import pipeline

# ============================================================
# 1. BIKE CATEGORIES
# ============================================================

BIKES = [
    "sports bike",
    "cruiser motorcycle",
    "dirt bike",
    "adventure motorcycle",
    "touring motorcycle",
    "cafe racer",
    "naked street bike",
    "electric motorcycle",
    "scooter",
    "moped",
    "mountain bicycle",
    "road bicycle",
    "hybrid bicycle",
    "BMX bicycle",
    "electric bicycle",
    "superbike",
    "standard motorcycle",
    "racing motorcycle",
    "motorcycle",
    "bicycle"
]

LABELS = [
    f"a photo of a {bike}"
    for bike in BIKES
]

# ============================================================
# 2. BIKE INFORMATION
# ============================================================

FACTS = {
    "sports bike": "Sports bikes are designed for speed, acceleration and performance.",
    "cruiser motorcycle": "Cruiser motorcycles are designed for comfortable riding and relaxed seating.",
    "dirt bike": "Dirt bikes are lightweight motorcycles designed for off-road riding.",
    "adventure motorcycle": "Adventure motorcycles are designed for both road and off-road travel.",
    "touring motorcycle": "Touring motorcycles are designed for long-distance comfortable journeys.",
    "cafe racer": "Cafe racers are lightweight motorcycles with a retro racing design.",
    "naked street bike": "Naked bikes have an exposed engine and an upright riding position.",
    "electric motorcycle": "Electric motorcycles use electric motors and rechargeable batteries.",
    "scooter": "Scooters generally have step-through frames and automatic transmissions.",
    "moped": "Mopeds are small, lightweight motorized two-wheelers.",
    "mountain bicycle": "Mountain bicycles are designed for rough terrain and off-road cycling.",
    "road bicycle": "Road bicycles are designed for speed on paved roads.",
    "hybrid bicycle": "Hybrid bicycles combine features of road and mountain bicycles.",
    "BMX bicycle": "BMX bicycles are designed for tricks, racing and stunt riding.",
    "electric bicycle": "Electric bicycles use battery-powered assistance while pedaling.",
    "superbike": "Superbikes are high-performance motorcycles with powerful engines.",
    "standard motorcycle": "Standard motorcycles offer an upright and practical riding position.",
    "racing motorcycle": "Racing motorcycles are designed for competitive track racing.",
    "motorcycle": "Motorcycles are two-wheeled motor-powered vehicles.",
    "bicycle": "Bicycles are human-powered two-wheeled vehicles."
}

# ============================================================
# 3. LOAD PRE-TRAINED AI MODEL
# ============================================================

print("Loading BikeVision AI...")

device = 0 if torch.cuda.is_available() else -1

print("Running on:", "GPU" if device == 0 else "CPU")

classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=device
)

print("BIKE AI MODEL LOADED SUCCESSFULLY!")

# ============================================================
# 4. DASHBOARD CSS
# ============================================================

CSS = """
.gradio-container {
    max-width: 1200px !important;
    background: #f1f5f9 !important;
}

#header {
    background: linear-gradient(120deg,#0f172a,#1e40af,#6d28d9);
    padding: 30px;
    border-radius: 18px;
    color: white;
    margin-bottom: 20px;
}

#header h1 {
    color: white !important;
    font-size: 34px;
}

#header p {
    color: #e0e7ff !important;
}

.result, .chart, .history, .stat {
    background: white;
    padding: 20px;
    border-radius: 15px;
    margin-bottom: 15px;
    color: #172554;
}

.result h1 {
    color: #1d4ed8;
    text-align: center;
}

.result h2, .result h3 {
    color: #1e3a8a;
}

.animal {
    font-size: 40px;
    text-align: center;
}

.score {
    color: #047857;
    font-size: 36px;
    font-weight: bold;
    text-align: center;
}

.muted {
    color: #475569;
    text-align: center;
}

.fact {
    background: #eff6ff;
    padding: 15px;
    border-radius: 10px;
    color: #1e3a8a;
}

.bar-row {
    margin: 15px 0;
}

.bar-label {
    display: flex;
    justify-content: space-between;
    margin-bottom: 6px;
    color: #172554;
}

.track {
    height: 12px;
    background: #cbd5e1;
    border-radius: 20px;
    overflow: hidden;
}

.fill {
    height: 100%;
    background: linear-gradient(90deg,#2563eb,#8b5cf6);
}

.history-row {
    display: flex;
    justify-content: space-between;
    gap: 10px;
    padding: 12px;
    border-bottom: 1px solid #e2e8f0;
    color: #172554;
}

#analyze {
    background: linear-gradient(100deg,#2563eb,#7c3aed);
    color: white;
    font-weight: bold;
}
"""

# ============================================================
# 5. BIKE RECOGNITION FUNCTION
# ============================================================

def recognize(image, history):

    if image is None:
        return (
            "<div class='result'>Please upload a bike image!</div>",
            "",
            pd.DataFrame(columns=["Rank", "Bike Category", "Score"]),
            "<p>No history available.</p>",
            history,
            None
        )

    image = image.convert("RGB")

    predictions = classifier(
        image,
        candidate_labels=LABELS
    )

    best = predictions[0]

    bike = best["label"].replace(
        "a photo of a ", ""
    )

    bike = bike.lower()

    score = float(best["score"]) * 100

    facts = FACTS.get(
        bike,
        "Explore more about this bike category."
    )

    # Result card

    result_html = f"""
    <div class="result">
        <div class="animal">🏍️</div>
        <p class="muted">PREDICTED BIKE CATEGORY</p>
        <h1>{html.escape(bike.upper())}</h1>
        <div class="score">{score:.2f}%</div>
        <p class="muted">Similarity Score</p>
        <hr>
        <h3>🏁 Bike Information</h3>
        <div class="fact">{html.escape(facts)}</div>
        <br>
        <small class="muted">
        Similarity scores are not calibrated probabilities.
        </small>
    </div>
    """

    # Top 5 predictions

    chart = "<div class='chart'><h3>📊 Top 5 Predictions</h3>"

    rows = []

    for i, item in enumerate(predictions[:5], 1):

        name = item["label"].replace(
            "a photo of a ", ""
        )

        percentage = float(item["score"]) * 100

        chart += f"""
        <div class="bar-row">
            <div class="bar-label">
                <span>{i}. {html.escape(name.title())}</span>
                <b>{percentage:.2f}%</b>
            </div>
            <div class="track">
                <div class="fill"
                style="width:{percentage:.2f}%"></div>
            </div>
        </div>
        """

        rows.append([
            i,
            name.title(),
            f"{percentage:.2f}%"
        ])

    chart += "</div>"

    table = pd.DataFrame(
        rows,
        columns=["Rank", "Bike Category", "Score"]
    )

    # Recognition history

    new_history = list(history or [])

    new_history.append({
        "Date": datetime.now().strftime("%d-%m-%Y %H:%M"),
        "Bike": bike.title(),
        "Score": f"{score:.2f}%"
    })

    history_html = """
    <div class="history">
    <h3>📚 Recognition History</h3>
    """

    for item in reversed(new_history[-10:]):

        history_html += f"""
        <div class="history-row">
            <span>🏍️ {html.escape(item['Bike'])}</span>
            <b>{html.escape(item['Score'])}</b>
            <small>{html.escape(item['Date'])}</small>
        </div>
        """

    history_html += "</div>"

    # Download report

    report = f"""
BIKEVISION AI REPORT
============================

Date: {datetime.now()}

Predicted Bike: {bike.title()}
Similarity Score: {score:.2f}%

TOP 5 PREDICTIONS
"""

    for row in rows:
        report += f"{row[0]}. {row[1]} - {row[2]}\n"

    report += f"\nBIKE INFORMATION\n{facts}\n"

    file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
        delete=False,
        encoding="utf-8"
    )

    file.write(report)
    file.close()

    return (
        result_html,
        chart,
        table,
        history_html,
        new_history,
        file.name
    )

# ============================================================
# 6. RESET FUNCTION
# ============================================================

def reset():

    return (
        None,
        "<div class='result'><h2>🏍️ Ready for Analysis</h2><p>Upload a bike image to begin.</p></div>",
        "",
        pd.DataFrame(columns=["Rank", "Bike Category", "Score"]),
        "<p>No history available.</p>",
        [],
        None
    )

# ============================================================
# 7. BUILD INTERACTIVE DASHBOARD
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="violet"
    ),
    css=CSS,
    title="BikeVision AI"
) as demo:

    history_state = gr.State([])

    gr.HTML("""
    <div id="header">
        <h1>🏍️ BikeVision AI</h1>
        <p>Artificial Intelligence Bike Image Classification Dashboard</p>
        <p>Powered by Hugging Face CLIP</p>
        <span>🟢 AI SYSTEM ONLINE</span>
    </div>
    """)

    with gr.Row():

        gr.HTML("""
        <div class="stat">
            <h2>CLIP</h2>
            <p>Pre-trained AI Model</p>
        </div>
        """)

        gr.HTML("""
        <div class="stat">
            <h2>20</h2>
            <p>Bike Categories</p>
        </div>
        """)

        gr.HTML("""
        <div class="stat">
            <h2>TOP 5</h2>
            <p>Predictions</p>
        </div>
        """)

    gr.Markdown("## 🔍 Bike Image Recognition")

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                label="Upload Bike Image",
                type="pil",
                height=350
            )

            analyze_btn = gr.Button(
                "🔍 ANALYZE IMAGE",
                variant="primary",
                elem_id="analyze"
            )

            reset_btn = gr.Button("↻ RESET DASHBOARD")

            gr.Markdown(
                "Supported formats: JPG, JPEG, PNG, WEBP"
            )

        with gr.Column():

            result_output = gr.HTML(
                "<div class='result'><h2>🏍️ Ready for Analysis</h2><p>Upload an image to begin.</p></div>"
            )

    gr.Markdown("---")
    gr.Markdown("## 📊 Prediction Analytics")

    chart_output = gr.HTML()

    prediction_table = gr.Dataframe(
        headers=["Rank", "Bike Category", "Score"],
        interactive=False,
        label="Top 5 Predictions"
    )

    gr.Markdown("---")
    gr.Markdown("## 📚 Recognition History")

    history_output = gr.HTML(
        "<p>No history available.</p>"
    )

    gr.Markdown("---")
    gr.Markdown("## 📥 Download Report")

    report_output = gr.File(
        label="Download Bike Recognition Report"
    )

    gr.HTML("""
    <div style="
        background:#0f172a;
        color:white;
        padding:25px;
        text-align:center;
        border-radius:12px;">
        <h3>🏍️ BIKEVISION AI</h3>
        Python | Gradio | Hugging Face | CLIP
    </div>
    """)

    # Analyze button

    analyze_btn.click(
        fn=recognize,
        inputs=[image_input, history_state],
        outputs=[
            result_output,
            chart_output,
            prediction_table,
            history_output,
            history_state,
            report_output
        ]
    )

    # Reset button

    reset_btn.click(
        fn=reset,
        inputs=[],
        outputs=[
            image_input,
            result_output,
            chart_output,
            prediction_table,
            history_output,
            history_state,
            report_output
        ]
    )

# ============================================================
# 8. LAUNCH APPLICATION
# ============================================================

print("Launching BikeVision AI...")

demo.queue()

demo.launch(
    share=True,
    debug=False,
    show_error=True
)

# ============================================================
# PROJECT COMPLETED
# ============================================================


Loading BikeVision AI...
Running on: CPU


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

BIKE AI MODEL LOADED SUCCESSFULLY!


/tmp/ipykernel_941/2550831059.py:382: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Launching BikeVision AI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://16b2daa13ac189967c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
